# Logit Lens Variants

Extended logit lens techniques beyond the basic version: contrastive lens
(compare two inputs), causal lens (with intervention), residual contribution lens,
and token trajectory tracking.

References:
- nostalgebraist (2020) "interpreting GPT: the logit lens"
- Belrose et al. (2023) "Eliciting Latent Predictions from Transformers with the Tuned Lens"

## Why This Matters

Logit lens variants analyzes how the model's output predictions form and change across layers. Understanding logit formation — which components contribute what to the final prediction — is central to explaining model behavior.

**Key references:**
- [nostalgebraist (2020) "interpreting GPT: the logit lens"](https://www.lesswrong.com/posts/AcKRB8wDpdaN6v6ru/interpreting-gpt-the-logit-lens) — Project intermediate residuals through the unembedding
- [Belrose et al. (2023) "Eliciting Latent Predictions"](https://arxiv.org/abs/2303.08112) — Learned affine probes improve on the logit lens

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.logit_lens_variants import (
    contrastive_logit_lens,
    causal_logit_lens,
    residual_contribution_lens,
    token_lens_trajectory,
    logit_lens_difference,
)

In [ ]:
cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([0, 5, 10, 15, 20, 25, 30])
print("Model ready")

## Contrastive Logit Lens

Compare logit lens predictions between two different inputs at each layer.
Shows where the model's intermediate representations diverge or converge.

In [ ]:
tokens_a = jnp.array([0, 5, 10, 15, 20, 25, 30])
tokens_b = jnp.array([1, 6, 11, 16, 21, 26, 31])
result = contrastive_logit_lens(model, tokens_a, tokens_b)
print(f"Divergence per layer:")
for i, d in enumerate(result['divergence_per_layer']):
    print(f"  Layer {i}: {d:.4f}")
print(f"\nConvergence layer: {result['convergence_layer']}")
print(f"\nTop divergent tokens (first 10):")
for layer, tok, diff in result['top_divergent_tokens'][:10]:
    print(f"  Layer {layer}, token {tok}: {diff:+.4f}")

## Causal Logit Lens

Apply an intervention at a specific layer and track how predictions change
at every downstream layer. Shows how interventions propagate.

In [ ]:
intervention = lambda x: x * 0.5  # halve activations
result = causal_logit_lens(model, tokens, intervention_layer=0, intervention_fn=intervention)
print(f"Prediction shifts per layer:")
for i, s in enumerate(result['prediction_shifts']):
    print(f"  Layer {i}: {s:.4f}")
print(f"\nFirst affected layer: {result['first_affected_layer']}")
print(f"\nClean vs intervened top predictions (per layer):")
for i in range(cfg.n_layers):
    print(f"  Layer {i}: clean={result['clean_predictions'][i][:3]}, "
          f"intervened={result['intervened_predictions'][i][:3]}")

## Residual Contribution Lens

Decompose logit lens predictions into per-component contributions. For each
layer, see how attention and MLP contribute to the logit lens prediction.

In [ ]:
result = residual_contribution_lens(model, tokens)
print(f"Attn logit contributions shape: {result['attn_logit_contributions'].shape}")
print(f"MLP logit contributions shape: {result['mlp_logit_contributions'].shape}")
print(f"\nTop attention-promoted tokens:")
sorted_attn = sorted(result['top_attn_tokens'], key=lambda x: -x[2])[:5]
for layer, tok, logit in sorted_attn:
    print(f"  Layer {layer}, token {tok}: {logit:.4f}")
print(f"\nTop MLP-promoted tokens:")
sorted_mlp = sorted(result['top_mlp_tokens'], key=lambda x: -x[2])[:5]
for layer, tok, logit in sorted_mlp:
    print(f"  Layer {layer}, token {tok}: {logit:.4f}")

## Token Lens Trajectory

Track specific tokens' probabilities through the logit lens at each layer.
See when target tokens emerge or get suppressed.

In [ ]:
result = token_lens_trajectory(model, tokens, target_tokens=[0, 1, 2])
print(f"Final prediction: token {result['final_prediction']}")
for tok in [0, 1, 2]:
    print(f"\nToken {tok}:")
    print(f"  Probs per layer: {[f'{p:.4f}' for p in result['token_probs'][tok]]}")
    print(f"  Ranks per layer: {result['token_ranks'][tok]}")
    print(f"  Emergence layer: {result['emergence_layers'][tok]}")

## Logit Lens Position Difference

Compare logit lens predictions between two positions within the same input.
Shows how position-specific information evolves through layers.

In [ ]:
result = logit_lens_difference(model, tokens, pos_a=0, pos_b=-1, top_k=5)
print(f"Logit difference per layer:")
for i, d in enumerate(result['logit_diff_per_layer']):
    print(f"  Layer {i}: {d:.4f}")
print(f"\nShared top-5 tokens per layer:")
for i, s in enumerate(result['shared_top_tokens']):
    print(f"  Layer {i}: {s}/5 shared")